**CI twin of `ch08-bias-variance.qmd`.** Generated by `infra/ci/make_twin.py` (P1-D10); do not edit by hand. It runs the chapter's worked example and exercise solutions under CPython so the R10 gate proves they work (DECISIONS D0010).

In [ ]:
# Make the single-sourced grader importable under CPython (no paste; P1-D9).
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_p / 'lib' / 'grader.py').exists():
        sys.path.insert(0, str(_p))
        break
from lib.grader import run_tests

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

def truth(x):
    return np.sin(2 * np.pi * x)

grid = np.linspace(0, 1, 101)
rng = np.random.default_rng(0)          # seeded: same panels every run

def train_panel(degree, n_sets=20, n_points=30):
    """Fit `n_sets` models of one family, each on its own noisy dataset."""
    curves = []
    for _ in range(n_sets):
        x = rng.uniform(0, 1, n_points)
        y = truth(x) + rng.normal(0, 0.3, n_points)
        pf = PolynomialFeatures(degree=degree, include_bias=False)
        m = LinearRegression().fit(pf.fit_transform(x.reshape(-1, 1)), y)
        curves.append(m.predict(pf.transform(grid.reshape(-1, 1))))
    return np.array(curves)

panel1, panel12 = train_panel(1), train_panel(12)

fig, axes = plt.subplots(1, 2, figsize=(7.6, 3), sharey=True)
for ax, curves, degree in [(axes[0], panel1, 1), (axes[1], panel12, 12)]:
    for c in curves:
        ax.plot(grid, c, lw=0.6, alpha=0.5)
    ax.plot(grid, truth(grid), "k--", lw=1.5, label="the truth")
    ax.set_ylim(-2, 2)
    ax.set_title(f"degree {degree}")
    ax.set_xlabel("x")
axes[0].legend(fontsize=8)
plt.show()

In [ ]:
for degree, curves in [(1, panel1), (12, panel12)]:
    mean_curve = curves.mean(axis=0)
    bias2 = ((mean_curve - truth(grid)) ** 2).mean()
    var = curves.var(axis=0).mean()
    print(f"degree {degree:2}:  bias² {bias2:.3f}   variance {var:.3f}   "
          f"sum {bias2 + var:.3f}")

In [ ]:
xprobe = np.linspace(0.01, 0.99, 200)

mean_errs = []
degrees = range(1, 13)
for degree in degrees:
    rng_u = np.random.default_rng(100)  # each family gets identical datasets
    errs = []
    for _ in range(20):
        x = rng_u.uniform(0, 1, 30)
        y = truth(x) + rng_u.normal(0, 0.3, 30)
        pf = PolynomialFeatures(degree=degree, include_bias=False)
        m = LinearRegression().fit(pf.fit_transform(x.reshape(-1, 1)), y)
        pred = m.predict(pf.transform(xprobe.reshape(-1, 1)))
        errs.append(((pred - truth(xprobe)) ** 2).mean())
    mean_errs.append(np.mean(errs))
    print(f"degree {degree:2}: mean test error {np.mean(errs):.3f}")

fig, ax = plt.subplots(figsize=(5.2, 3.2))
ax.plot(list(degrees), mean_errs, marker="o")
ax.set_yscale("log")
ax.set_xlabel("polynomial degree (capacity)")
ax.set_ylabel("mean squared test error")
ax.annotate("underfitting", (1.1, 0.16))
ax.annotate("sweet spot", (2.8, 0.012))
ax.annotate("overfitting", (9.3, 0.25))
plt.show()

In [ ]:
rows = [
    ("train high, validation high & close", "high bias (underfit)",
     "more capacity / better features"),
    ("train low, validation much worse", "high variance (overfit)",
     "leash it, shrink capacity, or more data"),
    ("validation swings across CV folds", "variance showing itself",
     "same levers as above"),
    ("both low, small gap", "near the sweet spot", "ship it; tune gently"),
]
print(f"{'symptom':38} {'diagnosis':26} lever")
print("-" * 92)
for symptom, diagnosis, lever in rows:
    print(f"{symptom:38} {diagnosis:26} {lever}")

In [ ]:
from lib.data import load_csv
from sklearn.linear_model import LinearRegression

homes = load_csv("california-housing-sample")
slopes = []
for seed in range(5):
    stack = homes.sample(n=30, random_state=seed)
    m = LinearRegression().fit(stack[["MedInc"]], stack["MedHouseVal"])
    slopes.append(round(float(m.coef_[0]), 3))

run_tests([
    ("five members, five slopes", slopes, [0.321, 0.387, 0.525, 0.503, 0.461]),
    ("the scatter is real", round(max(slopes) - min(slopes), 3), 0.204),
])

In [ ]:
def bias_squared(preds, truth):
    mean_pred = sum(preds) / len(preds)
    return (mean_pred - truth) ** 2

def variance(preds):
    mean_pred = sum(preds) / len(preds)
    return sum((p - mean_pred) ** 2 for p in preds) / len(preds)

run_tests([
    ("bias²: panel centred at 2.0, truth 3.0",
     bias_squared([2.0, 2.4, 1.6, 2.0], 3.0), 1.0),
    ("variance of that panel", variance([2.0, 2.4, 1.6, 2.0]), 0.08),
    ("a rigid panel has no variance", variance([2.5, 2.5, 2.5]), 0.0),
    ("an unbiased panel has no bias", bias_squared([2.0, 4.0], 3.0), 0.0),
    ("decomposition adds up: bias² + var = mean sq. error",
     bias_squared([2.0, 2.4, 1.6, 2.0], 3.0) + variance([2.0, 2.4, 1.6, 2.0]),
     sum((p - 3.0) ** 2 for p in [2.0, 2.4, 1.6, 2.0]) / 4),
], tol=1e-9)